In [73]:
import numpy as np
import numpy.linalg as lin
import qutip as qt
import math

In [ ]:
def sqrt(A):
    eigvals, eigvecs = lin.eig(A)
    eigvals = np.sqrt(eigvals)
    return np.matmul(np.matmul(eigvecs, np.diag(eigvals)), lin.inv(eigvecs))

def log(A):
    eigvals, eigvecs = lin.eig(A)
    eigvals = np.log(eigvals)
    return np.matmul(np.matmul(eigvecs, np.diag(eigvals)), lin.inv(eigvecs))

def exp(A):
    eigvals, eigvecs = lin.eig(A)
    eigvals = np.exp(eigvals)
    return np.matmul(np.matmul(eigvecs, np.diag(eigvals)), lin.inv(eigvecs))

In [131]:
def tr_x(p_xyz, dx, dy, dz):
    return np.trace(p_xyz.reshape(dx, dy, dz, dx, dy, dz), axis1=0, axis2=3)

def tr_y(p_xyz, dx, dy, dz):
    return np.trace(p_xyz.reshape(dx, dy, dz, dx, dy, dz), axis1=1, axis2=4)

def tr_z(p_xyz, dx, dy, dz):
    return np.trace(p_xyz.reshape(dx, dy, dz, dx, dy, dz), axis1=2, axis2=5)

def tr_yz(p_xyz, dx, dy, dz):
    p_xy = tr_z(p_xyz, dx, dy, dz).reshape(dx*dy, dx*dy)
    return np.trace(p_xy.reshape(dx, dy, dx, dy), axis1=1, axis2=3)

def tr_xz(p_xyz, dx, dy, dz):
    p_xy = tr_z(p_xyz, dx, dy, dz).reshape(dx*dy, dx*dy)
    return np.trace(p_xy.reshape(dx, dy, dx, dy), axis1=0, axis2=2)

def tr_xy(p_xyz, dx, dy, dz):
    p_xz = tr_y(p_xyz, dx, dy, dz)
    return np.trace(p_xz.reshape(dx, dz, dx, dz), axis1=0, axis2 = 2)

In [ ]:
def get_zlx(p_xyz, dx, dy, dz):
    p_xz = tr_y(p_xyz, dx, dy, dz)
    proj = np.kron(lin.inv(sqrt(tr_yz(p_xyz, dx, dy, dz))), np.eye(dz))
    return np.matmul(np.matmul(proj, p_xz), proj)

def get_zly(p_xyz, dx, dy, dz):
    p_yz = tr_x(p_xyz, dx, dy, dz)
    proj = np.kron(lin.inv(sqrt(tr_xz(p_xyz, dx, dy, dz))), np.eye(dz))
    return np.matmul(np.matmul(proj, p_yz), proj)
    

In [132]:
def R_c(p_cz, dc, dz, log_reg):
    return ((1-log_reg) * p_cz) + (log_reg * np.kron(np.eye(dc), np.eye(dz)/dz))

def R_z(p_z, dz, log_reg):
    return ((1-log_reg) * p_z) + (log_reg/dz * np.eye(dz))

def L_x(A, dx, dy, dz):
    A = np.kron(np.eye(dy), A)

    # Swap yxz to xyz
    A_1 = np.zeros(np.shape(A))
    for x in range(dx):
        for y in range(dy):
            for z in range(dz):
                for i in range(dx*dy*dz):
                    A_1[x*(dy*dz) + y*dz + z][i] = A[x*dz + y*(dx*dz) + z][i]
    
    A_2 = np.zeros(np.shape(A))
    for x in range(dx):
        for y in range(dy):
            for z in range(dz):
                for i in range(dx*dy*dz):
                    A_2[i][x*(dy*dz) + y*dz + z] = A_1[i][x*dz + y*(dx*dz) + z]

    return A_2

def L_y(A, dx):
    return np.kron(np.eye(dx), A)

def L_z(A, dx, dy):
    return np.kron(A, np.eye(dx*dy))

def condit_norm(R, dx, dy, dz):
    M_r = tr_z(R, dx, dy, dz)
    proj = np.kron(np.inv(sqrt(M_r)), np.eye(dz))
    return np.matmul(np.matmul(proj, R), proj)

In [ ]:
# Testing

# A = np.array([[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]]).reshape(4, 4)

# print(A)
#print(A.reshape(2, 2, 2, 2))
#print(np.trace(A.reshape(2,2,2,2), axis1=1, axis2=3))
# print(np.trace(A.reshape(2,2,2,2), axis1=0, axis2=1))

# psi = qt.tensor(qt.basis(2, 0), qt.basis(2, 1), (qt.basis(2, 0) + qt.basis(2, 1)).unit())
# x = np.kron(np.kron(np.array([[1, 0], [0, 0]]), np.array([[0, 0], [0, 1]])), np.array([[0.5, 0.5], [0.5, 0.5]]))

# print(psi.ptrace([0]))
# print(tr_yz(x, 2, 2, 2))

# print(np.exp(np.array([0, 1, 2])))

A = np.kron(np.kron(np.array([[0.4, 0.1], [0.6, 0.7]]), np.array([[1, 0], [0, 1]])), np.array([[0.2, 1.9], [3.5, 0]]))
A_1 = np.kron(np.array([[0.4, 0.1], [0.6, 0.7]]), np.array([[0.2, 1.9], [3.5, 0]]))
print(A)
print(L_x(A_1, 2, 2, 2))
print(np.array_equal(A, L_x(A_1, 2, 2, 2)))

In [ ]:
def initC(xy_size, dz):
    A = []

In [134]:
def QLatentSearch(esti_state, dx, dy, dz, smoothing, damping, log_reg, penalty, n):
    xy_size = esti_state.shape[0]
    smooth_esti = ((1-smoothing) * esti_state) + ((smoothing/xy_size) * np.eye(xy_size))
    C_zlxy = initC(xy_size, dz)
    for i in range(n):
        #Form p_xyz
        condit_projector = np.kron(sqrt(smooth_esti), np.eye(dz))
        p_xyz = np.matmul(np.matmul(condit_projector, C_zlxy), condit_projector)

        #Compute C_zlx, C_zly, & p_z
        C_zlx = get_zlx(p_xyz, dx, dy, dz)
        C_zly = get_zly(p_xyz, dx, dy, dz)
        p_z = tr_xy(p_xyz, dx, dy, dz)

        #Compute R
        Lx = L_x(log(R_c(C_zlx, dx, dz, log_reg)))
        Ly = L_y(log(R_c(C_zly, dy, dz, log_reg)))
        Lz = L_z(log(R_z(p_z, dz, log_reg)))
        R_xyz = exp(Lx + Ly + (penalty-1)*Lz)

        #Compute C~ and C
        C_norm = condit_norm(R_xyz)
        C_zlxy = ((1-damping)*C_zlxy) + (damping * C_norm)

    #Form p_xyz
        condit_projector = np.kron(sqrt(smooth_esti), np.eye(dz))
        p_xyz = np.matmul(np.matmul(condit_projector, C_zlxy), condit_projector)
    
    return p_xyz